# AlphaTrain pillar3b — V13 distillation with target sharpening

Next iteration past **pillar3a (sharp_25_epoch_12)**. Train on the V13 corpus generated by pillar3a + MCTS@200 (selfplay) and pillar3a + MCTS@600/400 (crisis). Apply the same target_temperature=0.5 sharpening recipe that gave +57% policy lift over baseline (HISTORY 153-156).

## Setup

| param | value | rationale |
|---|---|---|
| corpus | V13 (9.16M states) | selfplay 893 games + crisis 5,526 files |
| warm-start | pillar3a (sharp_25_epoch_12.pt) | strongest known policy |
| epochs | 17 | proven curve (pillar3a peaked at epoch 12) |
| batch-size | 32,768 | H100/G4 friendly |
| lr | 3e-4, warmup 1 epoch | |
| target_temperature | **0.5** | V13 top1 mean (T=1.0) = 0.249, matches V12's 0.26 → same recipe as pillar3a |
| color augmentation | on (default) | 7! symmetry — +4% lift (HISTORY 143-144) |
| dihedral augmentation | on (default) | 8× board symmetry |

Decision gate at eval time: ≥+30% over pillar3a mean (≥16,400 if pillar3a baseline is 12,622 policy; or proportional MCTS lift over 21,310 baseline).

## Upload to Drive (`MyDrive/alphatrain/`)

1. `colorlines_path_b.tar.gz` — code tarball (~1.2 MB)
2. `v13_pillar3a.pt.gz` — V13 corpus, gzipped (~613 MB)
3. `sharp_25_epoch_12.pt` — pillar3a warm-start (~143 MB; already on Drive from earlier pillar3a work)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, shutil, time

DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_path_b.tar.gz /content/
!cd /content && tar xzf colorlines_path_b.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
shutil.copy(f'{DRIVE}/sharp_25_epoch_12.pt',
            '/content/alphatrain/data/pillar3a.pt')
print(f'Warm-start (pillar3a): {os.path.getsize("/content/alphatrain/data/pillar3a.pt")/1e6:.0f} MB')

# Copy gzipped tensor to local disk FIRST, then verify size, then decompress.
# Streaming through Drive FUSE (`gzip -dc {DRIVE}/... > local`) is unreliable
# for files this size: FUSE can serve partial data and truncate the .pt silently.
t0 = time.time()
!cp {DRIVE}/v13_pillar3a.pt.gz /content/v13_pillar3a.pt.gz
gz_size = os.path.getsize('/content/v13_pillar3a.pt.gz')
print(f'.gz copied: {gz_size:,} bytes ({gz_size/1e6:.0f} MB)')
EXPECTED_GZ = 642_409_267
assert gz_size == EXPECTED_GZ, (
    f'.gz on Drive is truncated! got {gz_size} expected {EXPECTED_GZ}. '
    f'Re-upload v13_pillar3a.pt.gz to Drive before retrying.')
!gunzip -t /content/v13_pillar3a.pt.gz && echo '.gz integrity OK'
!gzip -dc /content/v13_pillar3a.pt.gz > /content/alphatrain/data/v13_pillar3a.pt
pt_size = os.path.getsize('/content/alphatrain/data/v13_pillar3a.pt')
EXPECTED_PT = 5_433_958_495
assert pt_size == EXPECTED_PT, (
    f'Decompressed .pt size wrong! got {pt_size} expected {EXPECTED_PT}.')
print(f'V13 tensor: {pt_size/1e9:.2f} GB ({time.time()-t0:.0f}s)')
# Optional: free disk by removing the .gz now that the .pt is verified.
!rm /content/v13_pillar3a.pt.gz

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

## Train pillar3b

Estimated wall time:
- H100: ~6h for 17 epochs at 9.16M states × 1.0 (with color × dihedral aug, effective 56× per state)
- G4: ~12h
- L4: ~10h

Watch the per-epoch `val_loss` (should drop monotonically through ~ep10, then plateau / slight wobble). Best checkpoint copied to Drive each epoch.

In [ ]:
%cd /content
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m alphatrain.train_path_b \
    --tensor-file alphatrain/data/v13_pillar3a.pt \
    --amp --compile \
    --resume alphatrain/data/pillar3a.pt --warm-start \
    --epochs 17 --batch-size 32768 --lr 3e-4 --warmup-epochs 1 \
    --target-temperature 0.5 \
    --copy-to /content/drive/MyDrive/alphatrain/pillar3b_best.pt \
    --save-dir /content/checkpoints/pillar3b

In [ ]:
# Copy every epoch checkpoint to Drive (for retrospective per-epoch eval)
import shutil, os, glob
DRIVE = '/content/drive/MyDrive/alphatrain'
RUN = 'pillar3b'
for f in sorted(glob.glob(f'/content/checkpoints/{RUN}/epoch_*.pt')):
    dst = f'{DRIVE}/{RUN}_{os.path.basename(f)}'
    shutil.copy(f, dst)
    print(f'Saved {dst} ({os.path.getsize(dst)/1e6:.0f} MB)')
for f in ['best.pt', 'latest.pt']:
    src = f'/content/checkpoints/{RUN}/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'{DRIVE}/{RUN}_{f}')
        print(f'Saved {DRIVE}/{RUN}_{f}')

## After pillar3b lands

Download `pillar3b_best.pt` (and intermediate epochs of interest) to M5. Then on M5:

1. Retrain value_head on pillar3b backbone (HISTORY 138): `python -m alphatrain.scripts.train_value_head --backbone alphatrain/data/pillar3b_best.pt --tensor-file alphatrain/data/v13_pillar3a.pt --output alphatrain/data/value_head_pillar3b.pt --epochs 5 --batch-size 8192 --lr 1e-3`
2. q_weight sweep on pillar3b + value_head_pillar3b (50 seeds, MCTS@100, cap=10K, q ∈ {1.5, 2.0, 2.5})
3. Compare to pillar3a at the SAME q_weight (apples-to-apples). Decision gate: ≥+30% mean over pillar3a.
4. If hit → pillar3c iteration on V14 (selfplay + crisis from pillar3b). If miss → try pillar3c with different temperature or value head.